In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from fraud_detect import config, data, features, viz, models

warnings.filterwarnings("ignore")
viz.configure_style()

## 9. Baseline Model — Logistic Regression

### Context
Before deploying complex gradient-boosting models, we establish a simple
logistic-regression baseline. This provides a reference point for measuring
improvement and helps identify which features matter in a linear framework.

### Objectives
- Train a logistic-regression model with proper preprocessing.
- Evaluate baseline performance using ROC-AUC.
- Identify top features by coefficient magnitude.
- Establish the performance floor for comparison.

> Utility code lives in the `fraud_detect` package (`src/fraud_detect/`).
> Install it in editable mode with `pip install -e .` before running.

In [ ]:
# Data loading
path = config.PROCESSED_TRAIN_PATH if config.PROCESSED_TRAIN_PATH.exists() else config.MERGED_TRAIN_PATH
train = pd.read_parquet(path)
print(f"Data loaded: {train.shape}")

In [3]:
# Stratified train/validation split via the package helper.
# Numeric feature selection automatically excludes TransactionID / isFraud / TransactionDT.

split = models.make_train_val_split(train)
print(f"Total:     {len(split.X_train) + len(split.X_val):,}")
print(f"Training:  {len(split.X_train):,} samples")
print(f"Validation:{len(split.X_val):,} samples")
print(f"Features:  {split.X_train.shape[1]}")

Total:     590,540
Training:  472,432 samples
Validation:118,108 samples
Features:  400


In [ ]:
# Build the pipeline (median impute -> standardise -> balanced logistic regression)
# and fit + evaluate in one step. The previous version called predict_proba
# without ever calling fit(), which raised NotFittedError.
#
# Note: n_jobs is intentionally omitted — LogisticRegression with the default
# lbfgs solver is single-threaded and recent sklearn versions reject the arg.

pipeline = models.build_logistic_pipeline(random_state=config.RANDOM_STATE)
scores = models.evaluate_classifier(pipeline, split)

print(f"Training ROC-AUC:   {scores['train_auc']:.4f}")
print(f"Validation ROC-AUC: {scores['val_auc']:.4f}")
print(f"Overfitting gap:    {scores['overfitting_gap']:.4f}")

In [ ]:
# Top features by absolute logistic-regression coefficient magnitude.
# Larger |coef| means stronger influence on the log-odds of fraud.

coef = pipeline.named_steps["model"].coef_[0]
top_coef = (
    pd.DataFrame({"feature": split.X_train.columns, "coef": coef})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(20)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)
top_coef